# EDA: Avito query–item data

Ноутбук является точкой входа к воспроизводимому анализу. Тяжёлые вычисления вынесены в `eda.py`, чтобы не дублировать логику и обрабатывать текстовые столбцы по одному.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, HTML, Markdown

ROOT = Path.cwd()
OUT = ROOT / 'eda_outputs'


## Полный пересчёт

Следующая ячейка заново профилирует три Parquet-файла и обновляет отчёты. Для обычного просмотра её запускать не нужно.


In [ ]:
%run eda.py


## Результаты последнего расчёта


In [ ]:
metrics = json.loads((OUT / 'metrics.json').read_text(encoding='utf-8'))
display(pd.DataFrame(metrics['profiles']['train']['columns_profile']))


# EDA: Avito query–item data

Дата расчёта: 2026-09-27 12:39  
Источник: `benchmark_items.parquet`, `benchmark_queries.parquet`, `train.parquet`.

## Краткий итог

- В данных **нет целевой переменной или оценки релевантности**. `train` содержит 497 673 наблюдений пар «сигнатура запроса — item_id» и выглядит как журнал положительных/наблюдавшихся взаимодействий. По этим файлам нельзя оценить баланс классов или построить честную supervised-валидацию без определения отрицательных примеров и метрики.
- В `train` 354 463 уникальных сигнатур запроса и 344 825 уникальных объявлений. 30 619 строк (6.15%) являются дополнительными наблюдениями уже встречавшейся пары.
- Категория запроса совпадает с категорией объявления в 99.989% строк; исключений всего 53. Локация совпадает в 83.10% строк — географическое несовпадение нельзя автоматически считать ошибкой.
- Пересечение benchmark с train неполное: 18 142 item_id (9.59% benchmark items), 108 полных сигнатур запросов (4.40%), но 915 текстов запросов (37.32%). Это сочетание transductive/cold-start сценариев.
- Есть сильный categorical shift: `search_category=0` составляет только 0.007% train, но 9.05% benchmark queries. Для этой группы итоговую метрику нужно считать отдельно.
- На выборке из 100 000 train-пар хотя бы один токен запроса встречается в заголовке у 71.99% пар; запрос целиком входит в заголовок у 35.35%. Одного lexical matching будет недостаточно.
- Пропуски рейтинга/числа отзывов вложены: во всех строках без числа отзывов также отсутствует рейтинг. Их нужно кодировать отдельным индикатором, а не только заполнять средним. Не найдено координат вне мирового диапазона; рейтинги лежат в [0, 5].

## 1. Объём и схема

| Таблица | Строк | Колонок | Размер | Row groups |
| --- | --- | --- | --- | --- |
| benchmark_items | 189 212 | 14 | 186.6 MB | 1 |
| benchmark_queries | 2 452 | 6 | 124.4 KB | 1 |
| train | 497 673 | 19 | 467.5 MB | 1 |

Все три файла состоят из одного row group. Для итеративной обработки это неудобно: чтение по батчам возможно, но predicate pushdown почти не помогает. Для обучения разумно один раз переписать подготовленные признаки в несколько row groups/партиций.

## 2. Пропуски и кардинальность

![Пропуски](eda_outputs/missingness.svg)

Ниже показаны только колонки, где есть `null`/`NaN` или пустые строки. Пустая строка не считается пропуском и приведена отдельно.

| Таблица | Колонка | Null/NaN | Доля | Пустые строки | Уникальных non-null |
| --- | --- | --- | --- | --- | --- |
| benchmark_items | item_rating | 17 631 | 9.32% | — | 6 007 |
| benchmark_items | item_rating_reviews_count | 11 615 | 6.14% | — | 1 170 |
| train | item_rating | 28 232 | 5.67% | — | 8 832 |
| train | item_rating_reviews_count | 19 211 | 3.86% | — | 1 372 |
| benchmark_items | item_description_raw | 35 | 0.02% | 0 | 168 134 |
| benchmark_items | item_longitude | 1 | 0.00% | — | 120 004 |
| benchmark_items | item_latitude | 1 | 0.00% | — | 119 778 |
| benchmark_queries | search_infm_params_text | 0 | 0.00% | 1 546 | 132 |
| train | search_infm_params_text | 0 | 0.00% | 163 999 | 3 724 |
| train | item_longitude | 4 | 0.00% | — | 228 567 |
| train | item_latitude | 4 | 0.00% | — | 227 858 |

Практический вывод: для текстовых полей следует различать `null` и пустую строку только если это имеет смысл в источнике; для модели их можно привести к единому специальному токену. Для рейтингов полезны пары признаков `rating_filled` + `rating_is_missing`.

## 3. Числовые признаки

| Таблица | Признак | N | Среднее | Медиана | p95 | p99 | Максимум |
| --- | --- | --- | --- | --- | --- | --- | --- |
| train | item_rating_reviews_count | 478 462 | 88.59 | 25.00 | 310.00 | 991.00 | 10 678.0 |
| train | item_rating | 469 441 | 4.82 | 5.00 | 5.00 | 5.00 | 5.00 |
| train | item_price | 497 673 | 19 711 227 | 1 000.0 | 10 000.0 | 99 000.0 | 999 999 999 999 |
| train | item_longitude | 497 669 | 49.40 | 40.93 | 92.85 | 128.14 | 177.52 |
| train | item_latitude | 497 669 | 53.88 | 55.17 | 59.97 | 64.53 | 71.64 |
| benchmark_items | item_rating_reviews_count | 177 597 | 66.48 | 18.00 | 249.00 | 796.00 | 18 291.0 |
| benchmark_items | item_rating | 171 581 | 4.79 | 5.00 | 5.00 | 5.00 | 5.00 |
| benchmark_items | item_price | 189 212 | 19 860 290 | 1 000.0 | 16 000.0 | 150 000.0 | 777 777 777 777 |
| benchmark_items | item_longitude | 189 211 | 48.22 | 39.74 | 91.45 | 131.90 | 177.52 |
| benchmark_items | item_latitude | 189 211 | 54.05 | 55.55 | 59.99 | 64.45 | 69.70 |

- Цена и число отзывов имеют длинный правый хвост. Средняя цена здесь неустойчива: максимум почти `10^12`, а значений выше 1 млрд — train 31, benchmark 8. Для модели нужны отдельный флаг специальных значений, `log1p` только для положительных цен и clip по train-квантилям.
- `item_price=-1` явно выглядит как sentinel: train — 32 631, benchmark — 15 248. Нулей отдельно: 4 995 и 1 340; их нельзя смешивать с настоящей ценой до уточнения семантики.
- Рейтингов вне [0, 5]: train — 0, benchmark — 0. Координат вне мировых границ нет, но пар `(0, 0)` — train 81, benchmark 61; это вероятный sentinel.

### 3.1. Категориальный сдвиг train → benchmark

| Срез | category=114 | Доля 114 | category=0 | Доля 0 | Другие |
| --- | --- | --- | --- | --- | --- |
| train.search_category | 497 635 | 99.992% | 34 | 0.007% | 4 |
| benchmark_queries.search_category | 2 230 | 90.95% | 222 | 9.05% | 0 |
| train.item_category_id | 497 658 | 99.997% | 0 | 0.000% | 15 |
| benchmark_items.item_category_id | 187 336 | 99.01% | 0 | 0.000% | 1 876 |

- `search_category=0`, вероятно, означает запрос без выбранной категории. Таких benchmark-запросов 222, а в train всего 34: это наиболее явный out-of-distribution сегмент.
- `search_is_delivery_search=1` встречается в train лишь 12 раз, а в benchmark отсутствует полностью. Этот флаг практически не обучаем и не проверяется текущим benchmark.
- Уникальных search locations: train 2 782, benchmark 273; item microcategories: 212 и 752 соответственно.

## 4. Текстовые поля

Длины указаны в Unicode-символах, не в токенах модели.

| Таблица | Поле | N | Медиана | p95 | p99 | Максимум | Пустых |
| --- | --- | --- | --- | --- | --- | --- | --- |
| benchmark_items | item_title_raw | 189 212 | 35.00 | 50.00 | 53.00 | 100.00 | 0 |
| benchmark_items | item_infm_params_text | 189 212 | 738.00 | 2 475.0 | 4 687.9 | 19 068.0 | 0 |
| benchmark_items | item_id | 189 212 | 16.00 | 16.00 | 16.00 | 16.00 | 0 |
| benchmark_items | item_description_raw | 189 177 | 1 054.0 | 4 205.2 | 5 541.0 | 8 494.0 | 0 |
| benchmark_queries | query_id | 2 452 | 16.00 | 16.00 | 16.00 | 16.00 | 0 |
| benchmark_queries | search_query | 2 452 | 22.00 | 39.00 | 47.49 | 70.00 | 0 |
| benchmark_queries | search_infm_params_text | 2 452 | 0.00 | 64.00 | 96.00 | 217.00 | 1 546 |
| train | search_query | 497 673 | 17.00 | 31.00 | 39.00 | 113.00 | 0 |
| train | search_infm_params_text | 497 673 | 28.00 | 97.00 | 106.00 | 1 079.0 | 163 999 |
| train | item_title_raw | 497 673 | 34.00 | 50.00 | 56.00 | 100.00 | 0 |
| train | item_infm_params_text | 497 673 | 871.00 | 2 965.0 | 6 504.6 | 11 899.0 | 0 |
| train | item_id | 497 673 | 16.00 | 16.00 | 16.00 | 16.00 | 0 |
| train | item_description_raw | 497 673 | 920.00 | 3 876.0 | 5 503.3 | 8 494.0 | 0 |

![Лексическое пересечение](eda_outputs/text_overlap.svg)

- Медианный recall токенов запроса по заголовку: 0.500; медианный Jaccard: 0.222.
- Для baseline имеет смысл сравнить BM25/TF-IDF по конкатенации title + attributes + description с bi-encoder. Длинные описания следует обрезать осознанно и измерить вклад отдельно.
- HTML/emoji/телефоноподобные строки и повторяющаяся пунктуация могут быть полезным сигналом качества объявления, поэтому агрессивно очищать текст до букв не стоит.

## 5. Структура train и повторы

| Показатель | Значение |
| --- | --- |
| Уникальных сырых текстов запроса | 74 529 |
| Уникальных нормализованных текстов запроса | 73 903 |
| Уникальных полных сигнатур запроса | 354 463 |
| Уникальных item_id | 344 825 |
| Уникальных query–item пар | 467 054 |
| Дополнительных повторов query–item | 30 619 |
| Групп повторяющихся query–item пар | 17 914 |
| Совпадение search/item category | 99.989% |
| Совпадение search/item location | 83.10% |

![Частоты](eda_outputs/frequency_distribution.svg)

Медиана — одно объявление на полную сигнатуру запроса и одно наблюдение на item_id, но максимумы равны соответственно 628.00 и 188.00. При случайном разбиении строк один и тот же запрос или item легко попадёт и в train, и в validation, завысив качество.

## 6. Benchmark и риск утечки

![Пересечение benchmark](eda_outputs/benchmark_overlap.svg)

| Проверка | Количество | Доля benchmark |
| --- | --- | --- |
| item_id встречался в train | 18 142 | 9.59% |
| Полная сигнатура запроса встречалась в train | 108 | 4.40% |
| Нормализованный текст запроса встречался в train | 915 | 37.32% |

Сам факт пересечения item_id не является ошибкой: benchmark может специально проверять ранжирование известного каталога. Но признаки, агрегированные по `item_id` или тексту запроса, должны строиться **только на train-части каждого фолда**. Нельзя заранее присоединять частоты/CTR, рассчитанные по всему `train`, к validation.

## 7. Стабильность атрибутов объявления

Часть item_id встречается многократно. Проверка ниже показывает, менялось ли значение атрибута внутри train.

| Атрибут | Item с >1 значением | Доля item | Максимум значений |
| --- | --- | --- | --- |
| item_title_raw | 0 | 0.000% | 1 |
| item_price | 0 | 0.000% | 1 |
| item_rating_reviews_count | 0 | 0.000% | 1 |
| item_rating | 0 | 0.000% | 1 |
| item_microcat_id | 0 | 0.000% | 1 |
| item_location_id | 0 | 0.000% | 1 |
| item_is_phone_hidden | 0 | 0.000% | 1 |
| item_is_message_forbidden | 0 | 0.000% | 1 |
| item_category_id | 0 | 0.000% | 1 |

Для 18 142 item_id, общих у train и benchmark, сопоставлены первые наблюдаемые значения:

| Атрибут | Совпало | Доля совпадений | Различается |
| --- | --- | --- | --- |
| item_title_raw | 18 142 | 100.00% | 0 |
| item_price | 18 142 | 100.00% | 0 |
| item_rating_reviews_count | 18 142 | 100.00% | 0 |
| item_rating | 18 142 | 100.00% | 0 |
| item_microcat_id | 18 142 | 100.00% | 0 |
| item_location_id | 18 142 | 100.00% | 0 |
| item_is_phone_hidden | 18 142 | 100.00% | 0 |
| item_is_message_forbidden | 18 142 | 100.00% | 0 |
| item_category_id | 18 142 | 100.00% | 0 |

Все выбранные атрибуты у 18 142 общих item_id совпали на 100%, а внутри train для одного item_id не найдено конфликтующих значений. Это указывает на единый snapshot каталога, а не историю изменений. Поэтому при валидации важно помнить: train-строка не содержит состояние объявления на момент реального взаимодействия — только предоставленный snapshot.

## 8. Рекомендации для моделирования и валидации

1. Сначала уточнить постановку: что считается релевантным, как формируются кандидаты и какая метрика используется (`Recall@K`, `NDCG@K`, `MRR` и т. п.). В текущих файлах нет label и benchmark ground truth.
2. Повторные одинаковые query–item наблюдения либо удалить для бинарной релевантности, либо превратить в вес/счётчик, если частота действительно отражает силу сигнала.
3. Делить данные группами, а не случайными строками. Подготовить как минимум два среза: unseen query signature и unseen item; отдельно измерять warm/cold performance и `search_category=0`.
4. Базовые признаки: нормализованный title/query, BM25/TF-IDF; category/microcategory; точное совпадение location; log-price/log-reviews; missingness рейтинга; флаги связи; длины текстов; token overlap.
5. Для semantic retrieval обучать bi-encoder с in-batch negatives и hard negatives из той же категории/локации. Любые отрицательные примеры нужно формировать так, чтобы необозначенный позитив не превратился автоматически в «истинный негатив».
6. Cross-encoder использовать как reranker ограниченного candidate set. Длинные descriptions и attributes лучше аблировать отдельно: они существенно увеличивают стоимость.
7. Зафиксировать дедупликацию, normalization и snapshot атрибутов до обучения; иначе offline-оценка будет нестабильной и может содержать утечку.

## Ограничения анализа

- Нельзя посчитать качество таргета, дисбаланс классов и корреляции с label: label отсутствует.
- Нет временной метки, поэтому невозможно проверить drift и сделать time split.
- Нет файла соответствий benchmark query–item с релевантностью, поэтому покрытие не равно качеству retrieval.
- Лексический overlap рассчитан на детерминированной выборке до 100 000 строк (`seed=42`); остальные сводки точные для всех строк.
